# Подготовка данных о бизнесах для аналитики

Цель: проверить и очистить данные о бизнесах, сохранив подготовленную таблицу для дальнейшего анализа.

## Часть 1. Подготовка ноутбука

Подключение библиотек, задание путей к файлам и настройка отображения таблиц и графиков.

### (1) Импорты

Импорт библиотек для работы с данными, расчётов и построения графиков.

In [71]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

### (2) Константы

Задание путей к исходному файлу и результату подготовки.

In [72]:
# Путь к заранее подготовленному CSV-файлу с бизнесами.
DATA_FILE = Path("data/businesses.csv")

# Файл с результатом подготовки.
OUTPUT_FILE = DATA_FILE.with_name("businesses_clean.csv")

### (3) Настройка отображения таблиц и графиков

Настройка компактного вывода таблиц и единого оформления графиков.

In [73]:
# Настройка отображения таблиц.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)
pd.set_option("display.min_rows", 10)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}".replace(",", " "))
pd.set_option("display.expand_frame_repr", False)

In [74]:
# Настройка отображения графиков.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.family": "DejaVu Sans",
})

## Часть 2. Подготовка данных: обработка и очистка

Проверка структуры, типов данных, пропусков, дубликатов, логической согласованности и категорий. Результат — подготовленная таблица с сохранёнными ID и статусами, поясняющими заполнение отсутствующих значений. Исходные данные хранятся отдельно в df_raw.

### (1) Загрузка и первый просмотр данных

Загрузка исходной таблицы, создание рабочей копии и просмотр первых 10 строк и размера данных.

In [75]:
# Исходная таблица и отдельная рабочая копия.
df_raw = pd.read_csv(DATA_FILE)
df = df_raw.copy()

print(f"Строк: {df.shape[0]:,}; столбцов: {df.shape[1]}")
display(df.head(10))

Строк: 9,754; столбцов: 28


,ID бизнеса,Тематика,Тип бизнеса (товары / услуги),Число подписчиков,Посты за 30 дней,Клипы за 30 дней,Посты с товаром за 30 дней,Клипы с товаром за 30 дней,Просмотры обычного контента,Лайки обычного контента,Репосты обычного контента,Просмотры товарного контента,Лайки товарного контента,Репосты товарного контента,Открытия карточки товара из контента,Товары загружены,Число активных товаров в каталоге,Подключён чекаут,Подключена доставка,Подключена оплата,Автор оригинального контента (оценка модели),Участие в реферальной программе,Заказы через чекаут,"GMV через чекаут, ₽","Средний чек, ₽","Траты на рекламу всего, ₽",Число активных услуг,"Медианная цена товара, ₽"
0,9308,Мода и стиль,без каталога,2800,66,1,22,1,1000000,51694,268,3800000,103250,1287,8372,0,0,0,0,0,1.000,1,0,0,NaN,0,0,NaN
1,3078,Спорт,без каталога,98000,7,46,0,7,550000,16429,10479,2300000,49299,28344,5818,0,0,1,0,0,1.000,1,0,0,NaN,0,0,NaN
2,5330,Образование,товарное,12000,380,0,26,0,380000,2530,636,1300000,1999,1065,6749,700,693,1,1,1,1.000,1,83,17150,208.000,0,0,169.000
3,5072,Развлечения и хобби,товарное,5200,12,0,20,0,260000,319,37,1200000,1380,1266,550,4986,4887,1,1,1,0.000,0,2,6250,2 648.000,0,0,1 325.000
4,6398,Личный блог,без каталога,440,35,20,7,12,530000,1915,455,1100000,2584,1228,10859,0,0,0,0,0,1.000,1,0,0,NaN,0,0,NaN
5,4267,Путешествия и отдых,товарное и услужное,22000,35,18,15,6,730000,2062,223,900000,1168,401,778,130,106,0,0,0,0.000,0,0,0,NaN,110000,31,4 240.000
6,2,Путешествия и отдых,товарное,1400,3,26,0,23,3600,338,1,540000,1754,566,1268,268,250,1,0,0,1.000,1,0,0,NaN,0,0,466.000
7,134,Образование,товарное,5800,0,2,0,1,5400,114,8,490000,1889,2036,796,28,28,1,1,0,0.000,0,0,0,NaN,0,0,20 140.000
8,4977,Творчество и хендмейд,товарное и услужное,8200,11,12,2,10,36000,38,10,460000,205,19,232,44,44,1,1,0,1.000,0,7,5350,756.000,0,4,530.000
9,2932,Личный блог,без каталога,5400,19,7,2,2,460000,2770,102,440000,925,475,3711,0,0,0,0,0,1.000,1,0,0,NaN,0,0,NaN


### (2) Проверка и преобразование типов

Проверка типов всех столбцов и преобразование тематики и типа бизнеса в текстовый тип string. Остальные столбцы сохраняют исходные типы, пропуски обрабатываются отдельно.

In [76]:
# Типы данных до преобразования.
display(df.dtypes.rename("Тип данных").to_frame())

,Тип данных
ID бизнеса,int64
Тематика,object
Тип бизнеса (товары / услуги),object
Число подписчиков,int64
Посты за 30 дней,int64
Клипы за 30 дней,int64
Посты с товаром за 30 дней,int64
Клипы с товаром за 30 дней,int64
Просмотры обычного контента,int64
Лайки обычного контента,int64


In [77]:
# Преобразование текстовых столбцов.
df = df.astype({
    "Тематика": "string",
    "Тип бизнеса (товары / услуги)": "string",
})

**Вывод:** столбцы «Тематика» и «Тип бизнеса (товары / услуги)» преобразованы из object в string для явного обозначения текстовых данных. Остальные исходные типы соответствуют содержимому столбцов и сохранены. Пропуски оставлены для отдельной обработки.

### (3) Первичный обзор данных

Описательная статистика числовых показателей и сводка текстовых категорий. Для чисел представлены количество заполненных значений, среднее, стандартное отклонение, минимум, квартили и максимум. ID исключён из расчётов. Для категорий представлены число заполненных значений (count), число уникальных значений (unique), наиболее частая категория (top) и количество её появлений (freq).

In [78]:
# Описательная статистика числовых показателей без ID.
display(df.drop(columns="ID бизнеса").describe().T)

,count,mean,std,min,25%,50%,75%,max
Число подписчиков,9 754.000,2 481.641,9 247.420,0.000,130.000,490.000,1 700.000,300 000.000
Посты за 30 дней,9 754.000,18.073,142.389,0.000,0.000,2.000,9.000,6 349.000
Клипы за 30 дней,9 754.000,2.337,13.767,0.000,0.000,0.000,1.000,792.000
Посты с товаром за 30 дней,9 754.000,1.724,79.581,0.000,0.000,0.000,0.000,4 400.000
Клипы с товаром за 30 дней,9 754.000,0.116,5.672,0.000,0.000,0.000,0.000,551.000
Просмотры обычного контента,9 754.000,32 416.201,414 890.990,0.000,320.000,1 300.000,5 600.000,29 000 000.000
Лайки обычного контента,9 754.000,346.678,4 112.807,0.000,1.000,12.000,75.750,200 478.000
Репосты обычного контента,9 754.000,137.957,3 632.183,0.000,0.000,0.000,4.000,219 350.000
Просмотры товарного контента,9 754.000,1 882.993,52 273.359,0.000,0.000,0.000,0.000,3 800 000.000
Лайки товарного контента,9 754.000,22.655,1 168.165,0.000,0.000,0.000,0.000,103 250.000


In [79]:
# Описательная статистика текстовых категорий.
display(df.select_dtypes(include=["object", "string"]).describe().T)

,count,unique,top,freq
Тематика,9754,42,Красота и здоровье,1703
Тип бизнеса (товары / услуги),9754,4,без каталога,3168


**Основные выводы первичного обзора**

- В таблице представлены 42 тематики и 4 типа бизнеса. Самая частая тематика — «Красота и здоровье» (1 703 бизнеса), самый частый тип — «без каталога» (3 168 бизнесов). Оба текстовых столбца заполнены полностью.
- Масштаб бизнесов существенно различается: медиана числа подписчиков — 490, среднее — около 2 482, максимум — 300 000. Для просмотров обычного контента медиана составляет 1 300, максимум — 29 000 000.
- Публикационная активность большинства бизнесов невысока: медиана числа постов за 30 дней — 2, третий квартиль — 9. Для клипов медиана равна 0, третий квартиль — 1. Это означает, что как минимум 75% бизнесов публикуют не более 9 постов и не более 1 клипа за 30 дней.
- Для постов и клипов с товаром третий квартиль равен 0: как минимум 75% бизнесов не публикуют контент соответствующего формата. При этом максимумы достигают 4 400 постов и 551 клипа с товаром. Высокие значения у небольшой части бизнесов повышают среднее, тогда как медиана определяется центральными значениями и значительно менее чувствительна к величине максимумов.
- У просмотров, лайков и репостов товарного контента, открытий карточек товара, заказов и GMV третий квартиль также равен нулю. Активность и результаты по этим показателям сосредоточены в меньшей части бизнесов.
- Рекламные расходы равны нулю как минимум у 75% бизнесов. При этом среднее составляет около 9 469 ₽, а максимум — 17 000 000 ₽. Среднее по всей таблице не описывает типичный бизнес, поскольку заметно зависит от крупных расходов небольшой части бизнесов.
- Чекаут подключён примерно у 58,8% бизнесов, доставка — у 9,4%, оплата — у 2,6%, участие в реферальной программе — у 2,1%. Для бинарных признаков среднее соответствует доле единиц.
- Число заполненных значений различается: оценка оригинального контента есть у 7 184 бизнесов, средний чек — у 291, медианная цена товара — у 4 342. Причины пропусков проверяются отдельно.
- Минимумы числовых показателей неотрицательны.

### (4) Описание столбцов

Описание показателей по смысловым блокам.

**Профиль бизнеса**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">ID бизнеса</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Идентификатор бизнеса, ключ таблицы.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Тематика</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Тематическая категория бизнеса, выбранная администратором при создании бизнес-сообщества. Преимущественно соответствует фактической тематике бизнеса.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Тип бизнеса (товары / услуги)</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Вид бизнеса, выбранный администратором: товарное, услужное, товарное и услужное, без каталога. Отсутствие каталога само по себе не означает отсутствие продаж.</td></tr>
</tbody>
</table>

**Аудитория сообщества**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Число подписчиков</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Размер аудитории сообщества по количеству подписчиков.</td></tr>
</tbody>
</table>

**Активность публикаций**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Посты за 30 дней</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество опубликованных постов за 30 дней.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Клипы за 30 дней</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество опубликованных клипов за 30 дней.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Посты с товаром за 30 дней</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество постов с товаром за 30 дней.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Клипы с товаром за 30 дней</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество клипов с товаром за 30 дней.</td></tr>
</tbody>
</table>

**Пометка:** посты за 30 дней и посты с товаром за 30 дней не пересекаются. Клипы за 30 дней и клипы с товаром за 30 дней также не пересекаются.

**Просмотры, реакции и переходы к товарам**

Контент включает посты и клипы. Показатели разделены на обычный контент и контент с товаром.

**Обычный контент**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Просмотры обычного контента</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество просмотров обычного контента.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Лайки обычного контента</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество лайков обычного контента.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Репосты обычного контента</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество репостов обычного контента.</td></tr>
</tbody>
</table>

**Товарный контент**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Просмотры товарного контента</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество просмотров товарного контента.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Лайки товарного контента</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество лайков товарного контента.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Репосты товарного контента</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество репостов товарного контента.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Открытия карточки товара из контента</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество открытий карточек товаров при переходе из контента.</td></tr>
</tbody>
</table>

**Пометка:** просмотры, лайки, репосты и открытия карточек отражают количество действий, а не число уникальных людей. Один пользователь может взаимодействовать с несколькими публикациями и карточками товаров. Действия за рассматриваемый месяц учитываются в том числе для контента, опубликованного в предыдущие периоды.

**Каталог и возможности покупки**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Товары загружены</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество загруженных товаров. Числовой показатель, а не признак 0/1.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Число активных товаров в каталоге</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество активных товаров в каталоге.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Число активных услуг</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество активных карточек услуг, предлагаемых бизнесом.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Подключён чекаут</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Подключение оформления заказа внутри VK: 1 — подключён, 0 — не подключён.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Подключена доставка</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Подключение доставки: 1 — подключена, 0 — не подключена.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Подключена оплата</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Подключение оплаты: 1 — подключена, 0 — не подключена.</td></tr>
</tbody>
</table>

**Пометка:** статус подключения чекаута отражает состояние на конец рассматриваемого периода. В течение периода чекаут мог быть включён, а к его концу — выключен. Поэтому значение 0 не исключает наличие заказов через чекаут за этот период.

**Оригинальный контент и реферальная программа**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Автор оригинального контента (оценка модели)</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Модельная оценка авторства оригинального контента: 1 — положительная, 0 — отрицательная. Пропуск означает отсутствие оценки.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Участие в реферальной программе</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Признак участия в программе: 1 — участвует, 0 — не участвует.</td></tr>
</tbody>
</table>

**Пометка:** участие в реферальной программе не означает, что по рекомендациям были совершены покупки или получен доход. В роли рекомендателя может выступать бизнес-сообщество, размещающее рекомендации товаров другого продавца. Это не определяет тип покупателя: им может быть физическое лицо.

**Продажи и рекламные расходы**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Заказы через чекаут</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Количество заказов, оформленных через чекаут. Не охватывает все возможные способы продажи.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">GMV через чекаут, ₽</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Суммарная стоимость заказов через чекаут за месяц в рублях.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Средний чек, ₽</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Средняя стоимость заказа через чекаут в рублях.</td></tr>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Траты на рекламу всего, ₽</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Общие рекламные расходы бизнеса в рублях.</td></tr>
</tbody>
</table>

**Пометка:** GMV расшифровывается как Gross Merchandise Value. Показатель приводится до вычета отмен, возвратов и комиссий. Расходы бизнеса из него не вычитаются; GMV не является прибылью.

**Ценовой уровень ассортимента**

<table style="width: 80%; max-width: 1000px; margin: 12px auto; table-layout: fixed;">
<thead><tr><th style="width: 35%; text-align: left;">Столбец</th><th style="text-align: left;">Описание</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Медианная цена товара, ₽</td><td style="text-align: left; vertical-align: top; overflow-wrap: anywhere;">Медиана цен товаров бизнеса в рублях. Описывает цены ассортимента, а не средний чек заказа.</td></tr>
</tbody>
</table>

### (5) Проверка и обработка пропусков

По результатам describe() пропуски есть только в трёх столбцах: «Автор оригинального контента (оценка модели)», «Средний чек, ₽» и «Медианная цена товара, ₽». Ниже представлены количество и доля пропусков в этих столбцах.

In [80]:
# Количество и доля пропусков только в незаполненных столбцах.
missing_summary = pd.DataFrame({
    "Количество пропусков": df.isna().sum(),
    "Доля пропусков, %": df.isna().mean() * 100,
})
missing_summary = missing_summary.loc[
    missing_summary["Количество пропусков"] > 0
].sort_values("Количество пропусков", ascending=False)

display(missing_summary)

,Количество пропусков,"Доля пропусков, %"
"Средний чек, ₽",9463,97.017
"Медианная цена товара, ₽",5412,55.485
Автор оригинального контента (оценка модели),2570,26.348


**Вывод:** пропуски сосредоточены в трёх столбцах: средний чек — 97,0%, медианная цена товара — 55,5%, оценка оригинального контента — 26,3%. Высокая доля пропусков не означает одинаковую проблему для всех строк: часть показателей может быть неприменима. Характер пропусков различается и рассматривается для каждого столбца отдельно.

#### (5.1) Средний чек

Проверка трёх ситуаций: отсутствие среднего чека при наличии заказов, отсутствие среднего чека при нулевых заказах и заполненный средний чек при нулевых заказах. Отсутствие среднего чека сопоставляется с наличием заказов.

In [81]:
# Наличие среднего чека относительно числа заказов.
display(pd.DataFrame({
    "Ситуация": [
        "Заказы есть, средний чек отсутствует",
        "Заказов нет, средний чек отсутствует",
        "Заказов нет, средний чек заполнен",
    ],
    "Число бизнесов": [
        ((df["Заказы через чекаут"] > 0) & df["Средний чек, ₽"].isna()).sum(),
        ((df["Заказы через чекаут"] == 0) & df["Средний чек, ₽"].isna()).sum(),
        ((df["Заказы через чекаут"] == 0) & df["Средний чек, ₽"].notna()).sum(),
    ],
}))

,Ситуация,Число бизнесов
0,"Заказы есть, средний чек отсутствует",7
1,"Заказов нет, средний чек отсутствует",9456
2,"Заказов нет, средний чек заполнен",0


**Вывод:** у 9 456 бизнесов отсутствуют и заказы через чекаут, и средний чек. Такие пропуски логически объяснимы отсутствием заказов. У 7 бизнесов заказы есть, но средний чек не заполнен. Заполненного среднего чека при нулевых заказах не обнаружено.

In [82]:
# Бизнесы с заказами без среднего чека и данные об ассортименте.
display(df.loc[
    (df["Заказы через чекаут"] > 0) & df["Средний чек, ₽"].isna(),
    [
        "ID бизнеса",
        "Тематика",
        "Тип бизнеса (товары / услуги)",
        "Заказы через чекаут",
        "GMV через чекаут, ₽",
        "Средний чек, ₽",
        "Товары загружены",
        "Число активных товаров в каталоге",
        "Медианная цена товара, ₽",
    ],
])

,ID бизнеса,Тематика,Тип бизнеса (товары / услуги),Заказы через чекаут,"GMV через чекаут, ₽","Средний чек, ₽",Товары загружены,Число активных товаров в каталоге,"Медианная цена товара, ₽"
462,4474,Творчество и хендмейд,товарное,1,0,NaN,43,43,159.000
772,1663,Творчество и хендмейд,товарное,4,0,NaN,8,8,716.000
4196,3307,Личный блог,товарное,2,0,NaN,1,1,NaN
4445,3588,Образование,товарное,1,0,NaN,283,283,106.000
5158,4412,Путешествия и отдых,товарное,1,0,NaN,2,2,3 975.000
6685,6175,Другое,товарное и услужное,4,0,NaN,2,2,1 060.000
9145,9036,Игры,без каталога,1,0,NaN,0,0,NaN


**Вывод:** у 7 бизнесов есть заказы через чекаут, но GMV равен нулю, а средний чек отсутствует. Причину такого сочетания по имеющимся данным установить нельзя. Бесплатность заказов и другие возможные объяснения не подтверждены.

In [83]:
# Нулевые значения среднего чека в исходных данных.
print("Количество нулевых чеков:", df_raw["Средний чек, ₽"].eq(0).sum())

Количество нулевых чеков: 0


**Решение:** отсутствие заказов и неизвестная стоимость при наличии заказов — разные случаи. Чтобы сохранить это различие, добавляется столбец «Статус среднего чека»: «Нет заказов», «Чек неизвестен» или «Чек указан». Статус определяется до заполнения по исходным данным. Затем пропуски заменяются на 0; его смысл определяется статусом, поэтому заполнение не означает бесплатную покупку.

In [84]:
# Статус для случаев, когда заказы есть, а чек неизвестен.
df["Статус среднего чека"] = "Чек неизвестен"

# Нулевое число заказов означает отсутствие заказов.
df.loc[df["Заказы через чекаут"] == 0, "Статус среднего чека"] = "Нет заказов"

# Исходно заполненный чек сохраняет статус «Чек указан».
# Исходные значения сопоставляются по индексам строк.
df.loc[
    df_raw.loc[df.index, "Средний чек, ₽"].notna(),
    "Статус среднего чека",
] = "Чек указан"

# Заполнение исходных пропусков нулём после определения статуса.
# Исходная таблица df_raw остаётся без изменений.
df["Средний чек, ₽"] = df_raw.loc[df.index, "Средний чек, ₽"].fillna(0)

# Число бизнесов с каждым статусом и проверка оставшихся пропусков.
display(df["Статус среднего чека"].value_counts().rename("Число бизнесов"))
print("Пропусков среднего чека:", df["Средний чек, ₽"].isna().sum())

Статус среднего чека
Нет заказов       9456
Чек указан         291
Чек неизвестен       7
Name: Число бизнесов, dtype: int64

Пропусков среднего чека: 0


**Итог:** 9 456 бизнесов имеют статус «Нет заказов», 7 — «Чек неизвестен», 291 — «Чек указан». В числовом столбце пропусков не остаётся. Для анализа среднего чека используются только строки «Чек указан», чтобы заполненные нули не искажали результаты. Все бизнесы сохраняются в рабочей таблице. Исходные значения остаются в df_raw.

#### (5.2) Медианная цена товара

Проверка пропусков относительно наличия загруженных и активных товаров.

In [85]:
# Наличие медианной цены относительно загруженных товаров.
display(pd.DataFrame({
    "Ситуация": [
        "Товаров нет, медианная цена отсутствует",
        "Товары есть, медианная цена отсутствует",
        "Товаров нет, медианная цена заполнена",
    ],
    "Число бизнесов": [
        (
            df["Товары загружены"].eq(0)
            & df["Медианная цена товара, ₽"].isna()
        ).sum(),
        (
            df["Товары загружены"].gt(0)
            & df["Медианная цена товара, ₽"].isna()
        ).sum(),
        (
            df["Товары загружены"].eq(0)
            & df["Медианная цена товара, ₽"].notna()
        ).sum(),
    ],
}))

,Ситуация,Число бизнесов
0,"Товаров нет, медианная цена отсутствует",4975
1,"Товары есть, медианная цена отсутствует",437
2,"Товаров нет, медианная цена заполнена",0


**Вывод:** у 4 975 бизнесов нет товаров и медианной цены, у 437 товары загружены, но цена отсутствует. Заполненной цены без загруженных товаров не обнаружено. Это разные случаи отсутствия цены; их смысл сохраняется при обработке.

In [86]:
# Примеры бизнесов с товарами без медианной цены.
display(df.loc[
    df["Товары загружены"].gt(0)
    & df["Медианная цена товара, ₽"].isna(),
    [
        "ID бизнеса",
        "Тематика",
        "Тип бизнеса (товары / услуги)",
        "Товары загружены",
        "Число активных товаров в каталоге",
        "Число активных услуг",
        "Медианная цена товара, ₽",
    ],
].head(10))

,ID бизнеса,Тематика,Тип бизнеса (товары / услуги),Товары загружены,Число активных товаров в каталоге,Число активных услуг,"Медианная цена товара, ₽"
13,2970,Мода и стиль,без каталога,254,0,0,NaN
56,6798,Технологии,без каталога,1,0,0,NaN
136,711,Путешествия и отдых,товарное и услужное,9,9,1,NaN
178,8670,Кулинария и еда,товарное,17,0,0,NaN
201,2552,Психология и отношения,товарное и услужное,1,1,4,NaN
264,5658,Игры,услужное,358,0,0,NaN
275,750,Товары и магазины,товарное,32,32,0,NaN
278,8023,Развлечения и хобби,товарное и услужное,2,0,25,NaN
331,1277,Недвижимость,товарное,49,40,0,NaN
342,7995,Дом и ремонт,товарное и услужное,4,4,0,NaN


**Активность каталога и отсутствие цены**

Сравнение долей пропусков среди бизнесов с загруженными товарами. Доля рассчитывается внутри каждой группы.

In [87]:
# Отбор бизнесов с загруженными товарами.
# Группировка по наличию активных товаров: True — есть, False — нет.
price_summary = (
    df.loc[df["Товары загружены"] > 0]
    .groupby(df["Число активных товаров в каталоге"] > 0)
    ["Медианная цена товара, ₽"]
    .agg(
        Бизнесов="size",
        Пропусков=lambda x: x.isna().sum(),
    )
)

# Доля пропусков от числа бизнесов внутри каждой группы.
price_summary["Доля пропусков, %"] = (
    price_summary["Пропусков"] / price_summary["Бизнесов"] * 100
)

# Понятные названия вместо True и False.
price_summary = price_summary.rename(index={
    True: "Есть активные товары",
    False: "Нет активных товаров",
})
price_summary.index.name = "Статус каталога"

display(price_summary)

,Бизнесов,Пропусков,"Доля пропусков, %"
Статус каталога,,,
Нет активных товаров,818,317,38.753
Есть активные товары,3961,120,3.030


**Вывод:** среди 818 бизнесов без активных товаров цена отсутствует у 317 (38,8%). Среди 3 961 бизнеса с активными товарами — у 120 (3,0%). При неактивном каталоге пропуски встречаются примерно в 13 раз чаще, но присутствуют и при активном. Эта связь не устанавливает точную причину отсутствия цены.

In [88]:
# Нулевые цены в исходных данных до заполнения пропусков.
print(
    "Исходных нулевых медианных цен:",
    df_raw["Медианная цена товара, ₽"].eq(0).sum(),
)

Исходных нулевых медианных цен: 3


**Решение:** добавляется столбец «Статус цены»: «Нет товаров», «Цена неизвестна» или «Цена указана». Статус определяется по исходным данным, затем пропуски цены заполняются нулём. Три исходные нулевые цены сохраняются со статусом «Цена указана», поэтому не смешиваются с заполненными значениями.

In [89]:
# Статус определяется по исходной цене до заполнения.
df["Статус цены"] = np.select(
    [
        df_raw.loc[df.index, "Медианная цена товара, ₽"].notna(),
        df["Товары загружены"].eq(0),
    ],
    ["Цена указана", "Нет товаров"],
    default="Цена неизвестна",
)

# Исходные нулевые цены сохраняют статус «Цена указана».
df["Медианная цена товара, ₽"] = (
    df_raw.loc[df.index, "Медианная цена товара, ₽"].fillna(0)
)

display(df["Статус цены"].value_counts().rename("Число бизнесов"))
print("Пропусков медианной цены:", df["Медианная цена товара, ₽"].isna().sum())

Статус цены
Нет товаров        4975
Цена указана       4342
Цена неизвестна     437
Name: Число бизнесов, dtype: int64

Пропусков медианной цены: 0


**Итог:** 4 975 бизнесов имеют статус «Нет товаров», 437 — «Цена неизвестна», 4 342 — «Цена указана». Пропусков в числовом столбце не остаётся. Для анализа цен используются только строки «Цена указана»: заполненные нули не влияют на среднее, медиану и распределение. Все бизнесы и исходные значения в df_raw сохраняются.

#### (5.3) Автор оригинального контента

Проверка количества и доли отсутствующих оценок модели.

In [90]:
# Количество и доля пропусков оценки в исходных данных.
print(
    "Пропусков оценки:",
    df_raw["Автор оригинального контента (оценка модели)"].isna().sum(),
)
print(
    "Доля пропусков:",
    f'{df_raw["Автор оригинального контента (оценка модели)"].isna().mean():.1%}',
)

Пропусков оценки: 2570
Доля пропусков: 26.3%


**Решение:** оценка отсутствует у 2 570 бизнесов (26,3%). Восстановить её по имеющимся данным нельзя. Значения 0 и 1 сохраняются, а пропуски заменяются на 2: 0 — отрицательная оценка, 1 — положительная, 2 — нет данных. Это коды категорий; 2 не означает оценку выше положительной.

In [91]:
# Сохранение исходных оценок 0/1; 2 обозначает отсутствие данных.
df["Автор оригинального контента (оценка модели)"] = (
    df_raw.loc[df.index, "Автор оригинального контента (оценка модели)"]
    .fillna(2)
    .astype("int64")
)

# Количество бизнесов с каждым кодом оценки.
display(
    df["Автор оригинального контента (оценка модели)"]
    .value_counts()
    .sort_index()
    .rename("Число бизнесов")
)

Автор оригинального контента (оценка модели)
0    5390
1    1794
2    2570
Name: Число бизнесов, dtype: int64

**Итог:** код 1 — у 1 794 бизнесов, 0 — у 5 390, 2 («Нет данных») — у 2 570. После преобразования пропусков в столбце не остаётся. Исходные значения сохраняются в df_raw.

#### (5.4) Итоговая проверка обработки пропусков

Повторная проверка трёх обработанных столбцов и общего числа пропусков в рабочей таблице.

In [92]:
# Пропуски в обработанных столбцах.
display(
    df[
        [
            "Средний чек, ₽",
            "Медианная цена товара, ₽",
            "Автор оригинального контента (оценка модели)",
        ]
    ]
    .isna()
    .sum()
    .rename("Осталось пропусков")
    .to_frame()
)

# Проверка всей рабочей таблицы.
print("Всего пропусков в df:", df.isna().sum().sum())

,Осталось пропусков
"Средний чек, ₽",0
"Медианная цена товара, ₽",0
Автор оригинального контента (оценка модели),0


Всего пропусков в df: 0


**Вывод:**

- **Средний чек:** пропуски заполнены нулями; различие между отсутствием заказов и неизвестным чеком сохранено в столбце «Статус среднего чека». Для анализа чека используются строки «Чек указан».
- **Медианная цена товара:** пропуски заполнены нулями; в столбце «Статус цены» разделены отсутствие товаров, неизвестная и указанная цена. Исходные нулевые цены сохранены как «Цена указана». Для анализа цен используются строки с этим статусом.
- **Автор оригинального контента:** исходные коды 0 и 1 сохранены, пропуски заменены на 2 («Нет данных»). Значения используются как коды категорий.

После выполнения обработки пропусков не остаётся. Все бизнесы сохранены, исходные данные остаются в df_raw.

### (6) Проверка дубликатов

Полный дубль — лишняя копия строки, в которой совпадают все значения. Повторяющийся ID — идентификатор бизнеса, который встречается больше одного раза.

In [93]:
# Полные дубли считаются без первого экземпляра строки.
print("Полных дублей:", df.duplicated().sum())

# Число разных ID, встречающихся больше одного раза.
print(
    "Повторяющихся ID:",
    df.loc[
        df["ID бизнеса"].duplicated(keep=False), "ID бизнеса"
    ].nunique(),
)

Полных дублей: 0
Повторяющихся ID: 0


**Вывод:** полных дублей и повторяющихся ID не обнаружено.

### (7) Проверка логической согласованности

Проверка допустимых значений и соотношений между показателями. Состояние каталога и инструментов на конец месяца рассматривается отдельно от активности и заказов за весь месяц.

In [94]:
# Количественные показатели должны быть целыми.
count_columns = [
    "Число подписчиков",
    "Посты за 30 дней",
    "Клипы за 30 дней",
    "Посты с товаром за 30 дней",
    "Клипы с товаром за 30 дней",
    "Просмотры обычного контента",
    "Лайки обычного контента",
    "Репосты обычного контента",
    "Просмотры товарного контента",
    "Лайки товарного контента",
    "Репосты товарного контента",
    "Открытия карточки товара из контента",
    "Товары загружены",
    "Число активных товаров в каталоге",
    "Заказы через чекаут",
    "Число активных услуг"
]

# Допустимые коды бинарных признаков.
binary_columns = [
    "Подключён чекаут",
    "Подключена доставка",
    "Подключена оплата",
    "Участие в реферальной программе"
]

In [95]:
# Отрицательные, бесконечные и дробные количественные значения.
numeric_checks = {
    "Отрицательные значения": (
        df.select_dtypes(include="number").lt(0).any(axis=1)
    ),
    "Бесконечные значения": (
        df.select_dtypes(include="number").isin([np.inf, -np.inf]).any(axis=1)
    ),
    "Дробные количества": (
        df[count_columns].mod(1).ne(0).any(axis=1)
    ),
}

display(pd.Series(
    {rule: mask.sum() for rule, mask in numeric_checks.items()},
    name="Число бизнесов",
).to_frame())

,Число бизнесов
Отрицательные значения,0
Бесконечные значения,0
Дробные количества,0


In [96]:
# Допустимы 0/1 для бинарных признаков и 0/1/2 для оценки контента.
code_checks = {
    "Недопустимые коды бинарных признаков": (
        ~df[binary_columns].isin([0, 1]).all(axis=1)
    ),
    "Недопустимая оценка оригинального контента": (
        ~df["Автор оригинального контента (оценка модели)"].isin([0, 1, 2])
    ),
}

display(pd.Series(
    {rule: mask.sum() for rule, mask in code_checks.items()},
    name="Число бизнесов",
).to_frame())

,Число бизнесов
Недопустимые коды бинарных признаков,0
Недопустимая оценка оригинального контента,0


In [97]:
# Активные товары должны быть частью загруженного ассортимента.
# Положительный GMV сопоставляется с наличием заказов.
relation_checks = {
    "Активных товаров больше загруженных": (
        df["Число активных товаров в каталоге"] > df["Товары загружены"]
    ),
    "Положительный GMV при отсутствии заказов": (
        (df["GMV через чекаут, ₽"] > 0) & (df["Заказы через чекаут"] == 0)
    ),
}

display(pd.Series(
    {rule: mask.sum() for rule, mask in relation_checks.items()},
    name="Число бизнесов",
).to_frame())

,Число бизнесов
Активных товаров больше загруженных,0
Положительный GMV при отсутствии заказов,0


**Вывод:** по выбранным проверкам нарушений не обнаружено: числовые показатели неотрицательны и конечны, количества целые, коды признаков допустимы. Активных товаров не больше загруженных; положительного GMV при нулевых заказах нет.

Заказы при выключенном чекауте, действия со старым контентом и отсутствие товаров при наличии услуг не считаются нарушениями. Равенство среднего чека отношению GMV к заказам и вложенность товарных публикаций в общее число не используются как обязательные условия.

### (8) Проверка уникальных значений категориальных признаков

Просмотр уникальных значений и частот для тематики и типа бизнеса.

In [98]:
# Уникальные значения тематики и типа бизнеса с частотами.
for column in ["Тематика", "Тип бизнеса (товары / услуги)"]:
    print(column)
    with pd.option_context("display.max_rows", None):
        display(
            df[column].value_counts(dropna=False)
            .rename("Число бизнесов")
            .to_frame()
        )

Тематика


,Число бизнесов
Тематика,
Красота и здоровье,1703
Кулинария и еда,837
Дом и ремонт,744
Творчество и хендмейд,538
Образование,537
Транспорт,521
Мода и стиль,460
Бизнес и работа,436
Путешествия и отдых,416


Тип бизнеса (товары / услуги)


,Число бизнесов
Тип бизнеса (товары / услуги),
без каталога,3168
товарное,3150
услужное,2428
товарное и услужное,1008


**Вывод:** представлены 42 тематики и 4 типа бизнеса. При визуальном просмотре явных повторов категорий с разным написанием и некорректных названий не обнаружено. Значения сохранены без изменений.

### (9) Итоги подготовки данных

В ходе подготовки проверены типы данных, пропуски, дубликаты, логическая согласованность показателей и значения категориальных признаков. Пропуски обработаны с сохранением их смысла: для среднего чека и медианной цены добавлены статусы, отсутствие оценки оригинального контента обозначено отдельным кодом.

Все бизнесы сохранены, исходная таблица остаётся без изменений. Подготовленная таблица готова к сохранению и дальнейшему исследованию в аналитическом ноутбуке.

## Часть 3. Сохранение подготовленных данных

Сохранение рабочей таблицы в отдельный CSV-файл для загрузки в аналитическом ноутбуке. Исходный файл businesses.csv остаётся без изменений.

### (1) Сохранение файла

В файл включаются подготовленные значения и статусы среднего чека и цены. Индекс строк не сохраняется; ключом остаётся «ID бизнеса».

In [ ]:
# Сохранение таблицы в UTF-8 без индекса.
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(OUTPUT_FILE)
print(f"Строк: {df.shape[0]:,}; столбцов: {df.shape[1]}")